# DJI Avata 360 — 8K Equirectangular Person Detection

Processes the stitched 7680×3840 equirectangular export from DJI Studio.
Extracts perspective views and runs YOLO person detection across the full descent.

**Input:** `Nakkila360.mp4` (7680×3840, 50fps, 197s) on Google Drive

**Comparison:** LRF fisheye pipeline (960px) vs 8K equirectangular (7680×3840)

In [ ]:
#@title 1. Setup
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

import cv2, numpy as np, torch, json, time
from ultralytics import YOLO

print('GPU:', torch.cuda.get_device_name(0))

# Load video
VIDEO = '/content/drive/MyDrive/DroneCV/Nakkila360.mp4'
cap = cv2.VideoCapture(VIDEO)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'Video: {w}x{h} @ {fps:.0f}fps, {total} frames ({total/fps:.0f}s)')

# Load model
model = YOLO('/content/drive/MyDrive/DroneCV/results/visdrone_yolov8m_1280_best.pt')
print('Model loaded')

In [ ]:
#@title 2. Equirectangular → Perspective Extraction Function

def equirect_to_perspective(equirect, yaw_deg=0, pitch_deg=0, fov=90, out_size=(1280, 960)):
    """Extract rectilinear perspective view from equirectangular 360 frame.
    
    Args:
        equirect: 7680x3840 equirectangular frame
        yaw_deg: horizontal rotation (0=front, 90=right, 180=back, 270=left)
        pitch_deg: vertical angle (0=horizon, negative=look down, positive=look up)
        fov: field of view in degrees
        out_size: output (width, height)
    """
    out_w, out_h = out_size
    f = out_w / (2 * np.tan(np.radians(fov / 2)))
    u = np.arange(out_w, dtype=np.float64) - out_w / 2
    v = np.arange(out_h, dtype=np.float64) - out_h / 2
    u, v = np.meshgrid(u, v)
    x, y, z = u, v, np.full_like(u, f)
    norm = np.sqrt(x**2 + y**2 + z**2)
    x, y, z = x/norm, y/norm, z/norm
    # Pitch (around x-axis)
    cp, sp = np.cos(np.radians(pitch_deg)), np.sin(np.radians(pitch_deg))
    y, z = cp*y - sp*z, sp*y + cp*z
    # Yaw (around y-axis)
    cy, sy = np.cos(np.radians(yaw_deg)), np.sin(np.radians(yaw_deg))
    x, z = cy*x + sy*z, -sy*x + cy*z
    # To equirectangular pixel coords
    lon = np.arctan2(x, z)
    lat = np.arcsin(np.clip(y, -1, 1))
    h_eq, w_eq = equirect.shape[:2]
    src_x = ((lon / np.pi + 1) / 2 * w_eq).astype(np.float32)
    src_y = ((0.5 - lat / np.pi) * h_eq).astype(np.float32)
    return cv2.remap(equirect, src_x, src_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)

print('Extraction function ready')
print('Orientation: pitch=0 is horizon, pitch<0 looks down, pitch>0 looks up')

In [ ]:
#@title 3. Full Descent Comparison (every 1s from t=160 to t=196)
from ultralytics import YOLO
import time

# Also load COCO for close-range
coco = YOLO('yolov8s.pt')

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)

results = []
t0 = time.time()

print(f'{"t":>4} | {"v8m":>5} | {"COCO":>5} | {"best":>5} | pitch | yaw')
print('-' * 55)

for t in range(160, 197):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(t * fps))
    ret, frame = cap.read()
    if not ret:
        continue
    
    best_v8m, best_coco = 0, 0
    best_p, best_y = 0, 0
    
    for pitch in [-5, -10, -15, -20, -30, -50, -70]:
        for yaw in range(0, 360, 30):
            view = equirect_to_perspective(frame, yaw_deg=yaw, pitch_deg=pitch)
            r1 = model(view, conf=0.2, classes=[0, 1], imgsz=1280, verbose=False)[0]
            r2 = coco(view, conf=0.2, classes=[0], imgsz=640, verbose=False)[0]
            c1 = max(float(b.conf) for b in r1.boxes) if len(r1.boxes) > 0 else 0
            c2 = max(float(b.conf) for b in r2.boxes) if len(r2.boxes) > 0 else 0
            if c1 > best_v8m:
                best_v8m = c1
            if c2 > best_coco:
                best_coco = c2
                best_p, best_y = pitch, yaw
    
    best = max(best_v8m, best_coco)
    results.append({'t': t, 'v8m': round(best_v8m, 3), 'coco': round(best_coco, 3), 
                    'best': round(best, 3), 'pitch': best_p, 'yaw': best_y})
    print(f'{t:>4} | {best_v8m:>5.2f} | {best_coco:>5.2f} | {best:>5.2f} | {best_p:>5} | {best_y:>3}')

cap.release()
elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min')

# Save results
import json
with open('/content/drive/MyDrive/DroneCV/results/equirect_8k_descent.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved: equirect_8k_descent.json')

In [ ]:
#@title 4. Summary — Compare LRF vs 8K Equirectangular
import json

with open('/content/drive/MyDrive/DroneCV/results/equirect_8k_descent.json') as f:
    results_8k = json.load(f)

# LRF baseline (from previous evaluation)
lrf_baseline = {183: 0.63, 186: 0.77, 188: 0.48, 190: 0.50, 194: 0.0}  # combined model on LRF

print('Comparison: LRF fisheye (960px) vs 8K equirect (7680x3840)')
print(f'{"t":>4} | {"LRF":>5} | {"8K":>5} | {"diff":>5}')
print('-' * 30)

for r in results_8k:
    t = r['t']
    if t in lrf_baseline:
        lrf = lrf_baseline[t]
        diff = r['best'] - lrf
        print(f'{t:>4} | {lrf:>5.2f} | {r["best"]:>5.2f} | {diff:>+5.2f}')

detected = sum(1 for r in results_8k if r['best'] > 0.3)
print(f'\n8K: Person detected in {detected}/{len(results_8k)} frames (conf>0.3)')